![dvd_image](dvd_image.jpg)

Une entreprise de location de DVD a besoin de votre aide ! Elle souhaite estimer combien de jours un client va conserver un DVD en fonction de certaines caractéristiques et s’est tournée vers vous. Votre mission : tester quelques modèles de régression pour prédire le nombre de jours de location. L’entreprise vise un modèle dont la MSE sur un jeu de test est inférieure ou égale à 3. Le modèle que vous construirez aidera l’entreprise à mieux planifier ses stocks.

Les données sont fournies dans le fichier CSV `rental_info.csv`. Il contient les variables suivantes :
- `"rental_date"` : la date (et l’heure) à laquelle le client loue le DVD.
- `"return_date"` : la date (et l’heure) à laquelle le client rend le DVD.
- `"amount"` : le montant payé par le client pour la location du DVD.
- `"amount_2"` : le carré de `"amount"`.
- `"rental_rate"` : le tarif de location du DVD.
- `"rental_rate_2"` : le carré de `"rental_rate"`.
- `"release_year"` : l’année de sortie du film loué.
- `"length"` : la durée du film loué, en minutes.
- `"length_2"` : le carré de `"length"`.
- `"replacement_cost"` : le coût de remplacement du DVD pour l’entreprise.
- `"special_features"` : les bonus éventuels (par exemple, bandes‑annonces/scènes coupées) inclus sur le DVD.
- `"NC-17"`, `"PG"`, `"PG-13"`, `"R"` : variables indicatrices (dummies) de la classification du film. La valeur est 1 si le film a la classification indiquée par le nom de la colonne, sinon 0. Pour votre commodité, la variable de référence a déjà été supprimée.

In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Modèles à tester
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# -----------------------------
# 1. Chargement des données
# -----------------------------
df = pd.read_csv("rental_info.csv")

# -----------------------------
# 2. Création de rental_length_days
# -----------------------------
df["rental_date"] = pd.to_datetime(df["rental_date"])
df["return_date"] = pd.to_datetime(df["return_date"])

df["rental_length_days"] = (df["return_date"] - df["rental_date"]).dt.days

# -----------------------------
# 3. Création des dummies Deleted Scenes / Behind the Scenes
# -----------------------------
df["deleted_scenes"] = df["special_features"].str.contains("Deleted Scenes", na=False).astype(int)
df["behind_the_scenes"] = df["special_features"].str.contains("Behind the Scenes", na=False).astype(int)

# -----------------------------
# 4. Construction de X et y
# -----------------------------
cols_to_exclude = [
    "rental_length_days",   # cible
    "return_date",          # fuite d'information
    "rental_date",          # fuite d'information
    "special_features"      # remplacée par dummies
]

X = df.drop(columns=cols_to_exclude)
y = df["rental_length_days"]

# -----------------------------
# 5. Split train/test
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=9
)

# -----------------------------
# 6. Test de plusieurs modèles
# -----------------------------
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.01),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=9),
    "GradientBoosting": GradientBoostingRegressor(random_state=9)
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    results[name] = mse
    print(f"{name} → MSE = {mse:.3f}")

# -----------------------------
# 7. Sélection du meilleur modèle ≤ 3
# -----------------------------
valid_models = {name: mse for name, mse in results.items() if mse <= 3}

if len(valid_models) == 0:
    print("Aucun modèle n'a une MSE ≤ 3.")
    best_model = None
    best_mse = None
else:
    best_name = min(valid_models, key=valid_models.get)
    best_mse = valid_models[best_name]
    best_model = models[best_name]

    print(f"\nMeilleur modèle : {best_name}")
    print(f"MSE : {best_mse:.3f}")


LinearRegression → MSE = 2.942
Ridge → MSE = 2.942
Lasso → MSE = 2.950
RandomForest → MSE = 2.028
GradientBoosting → MSE = 2.425

Meilleur modèle : RandomForest
MSE : 2.028
